In [ ]:
import requests
import json

# Set the base URL for 4chan API
board_url = 'https://a.4cdn.org/lgbt/'
thread_url = 'https://a.4cdn.org/lgbt/thread/'

# Function to scrape a thread using the 4chan API
def scrape_thread_api(thread_id):
    try:
        # Construct the API URL for the thread
        api_url = f'{thread_url}{thread_id}.json'

        # Send an HTTP request to fetch the thread
        response = requests.get(api_url)
        response.raise_for_status()

        # Parse the JSON response
        thread_data = response.json()

        # Extract the original post (OP) and replies
        op_post = thread_data['posts'][0]  # First post is the OP
        op_text = op_post.get('com', 'No original post content.')

        replies = []
        for post in thread_data['posts'][1:]:  # Skip the first post (OP)
            reply_text = post.get('com', '')
            replies.append(reply_text)

        # Output the scraped data
        return {
            'thread_id': thread_id,
            'original_post': op_text,
            'replies': replies
        }

    except requests.RequestException as e:
        print(f"Error fetching thread {thread_id}: {e}")
        return None

# Function to get list of active threads on the /lgbt/ board
def get_active_threads():
    try:
        # Send request to get active threads on the /lgbt/ board
        response = requests.get(f'{board_url}threads.json')
        response.raise_for_status()

        # Parse the JSON response
        board_data = response.json()

        # Extract all thread IDs (limit to 100 threads)
        thread_ids = []
        for page in board_data:
            for thread in page['threads']:
                thread_ids.append(thread['no'])  # 'no' is the thread ID
                if len(thread_ids) >= 100:  # Stop after 100 threads
                    return thread_ids

        return thread_ids

    except requests.RequestException as e:
        print(f"Error fetching active threads: {e}")
        return []

# Main function to scrape multiple threads
def scrape_multiple_threads():
    # Get list of up to 100 active thread IDs
    thread_ids = get_active_threads()

    if not thread_ids:
        print("No threads found to scrape.")
        return

    # Scrape each thread and collect data
    all_scraped_data = []
    for thread_id in thread_ids:
        scraped_data = scrape_thread_api(thread_id)
        if scraped_data:
            all_scraped_data.append(scraped_data)

    # Output all scraped data as JSON
    print(json.dumps(all_scraped_data, indent=4))

# Run the scraper for multiple threads
scrape_multiple_threads()


In [ ]:
import requests
import csv
import time
from datetime import datetime

# Set the base URL for 4chan API
board_url = 'https://a.4cdn.org/lgbt/'
thread_url = 'https://a.4cdn.org/lgbt/thread/'

# Store timestamps for "If-Modified-Since" headers
last_modified_times = {}

# Function to scrape a thread using the 4chan API
def scrape_thread_api(thread_id):
    global last_modified_times

    try:
        # Construct the API URL for the thread
        api_url = f'{thread_url}{thread_id}.json'

        # Prepare headers with If-Modified-Since
        headers = {}
        if thread_id in last_modified_times:
            headers['If-Modified-Since'] = last_modified_times[thread_id]

        # Send an HTTP request to fetch the thread
        response = requests.get(api_url, headers=headers)

        # If status code is 304 (Not Modified), skip this thread
        if response.status_code == 304:
            print(f"Thread {thread_id} has not been modified since the last request.")
            return None

        # Raise an error if the request was unsuccessful
        response.raise_for_status()

        # Parse the JSON response
        thread_data = response.json()

        # Update the last modified time for this thread
        if 'Last-Modified' in response.headers:
            last_modified_times[thread_id] = response.headers['Last-Modified']

        # Extract the original post (OP) and replies
        op_post = thread_data['posts'][0]  # First post is the OP
        op_text = op_post.get('com', 'No original post content.')

        replies = []
        for post in thread_data['posts'][1:]:  # Skip the first post (OP)
            reply_text = post.get('com', '')
            replies.append(reply_text)

        # Output the scraped data
        return {
            'thread_id': thread_id,
            'original_post': op_text,
            'replies': replies
        }

    except requests.RequestException as e:
        print(f"Error fetching thread {thread_id}: {e}")
        return None

# Function to get list of active threads on the /lgbt/ board
def get_active_threads():
    try:
        # Send request to get active threads on the /lgbt/ board
        response = requests.get(f'{board_url}threads.json')
        response.raise_for_status()

        # Parse the JSON response
        board_data = response.json()

        # Extract all thread IDs (limit to 100 threads)
        thread_ids = []
        for page in board_data:
            for thread in page['threads']:
                thread_ids.append(thread['no'])  # 'no' is the thread ID
                if len(thread_ids) >= 100:  # Stop after 100 threads
                    return thread_ids

        return thread_ids

    except requests.RequestException as e:
        print(f"Error fetching active threads: {e}")
        return []

# Function to save scraped data to CSV
def save_to_csv(scraped_data, filename='scraped_threads.csv'):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        # Write the header
        writer.writerow(['Thread ID', 'Post Type', 'Content'])

        # Write data for each thread
        for data in scraped_data:
            # Write the original post
            writer.writerow([data['thread_id'], 'Original Post', data['original_post']])

            # Write each reply
            for reply in data['replies']:
                writer.writerow([data['thread_id'], 'Reply', reply])

    print(f"Data saved to {filename}")

# Main function to scrape multiple threads and save to CSV
def scrape_multiple_threads_and_save():
    # Get list of up to 100 active thread IDs
    thread_ids = get_active_threads()

    if not thread_ids:
        print("No threads found to scrape.")
        return

    # Scrape each thread and collect data
    all_scraped_data = []
    for thread_id in thread_ids:
        scraped_data = scrape_thread_api(thread_id)
        if scraped_data:
            all_scraped_data.append(scraped_data)

        # Delay to ensure no more than 1 request per second
        time.sleep(1)

    # Save the scraped data to a CSV file
    save_to_csv(all_scraped_data)

# Run the scraper for multiple threads and save the data
scrape_multiple_threads_and_save()

# Add an update function for thread monitoring (optional)
def update_threads():
    while True:
        print("Updating threads...")
        scrape_multiple_threads_and_save()

        # Wait for 10 seconds or more before updating again
        time.sleep(10)

# Uncomment below to start the update loop for continuous monitoring
# update_threads()


In [ ]:
import requests
import csv
import time
from datetime import datetime

# Set the base URL for 4chan API
board_url = 'https://a.4cdn.org/lgbt/'
thread_url = 'https://a.4cdn.org/lgbt/thread/'
archive_url = 'https://a.4cdn.org/lgbt/archive.json'

# Store timestamps for "If-Modified-Since" headers
last_modified_times = {}

# Function to scrape a thread using the 4chan API
def scrape_thread_api(thread_id):
    global last_modified_times

    try:
        # Construct the API URL for the thread
        api_url = f'{thread_url}{thread_id}.json'

        # Prepare headers with If-Modified-Since
        headers = {}
        if thread_id in last_modified_times:
            headers['If-Modified-Since'] = last_modified_times[thread_id]

        # Send an HTTP request to fetch the thread
        response = requests.get(api_url, headers=headers)

        # If status code is 304 (Not Modified), skip this thread
        if response.status_code == 304:
            print(f"Thread {thread_id} has not been modified since the last request.")
            return None

        # Raise an error if the request was unsuccessful
        response.raise_for_status()

        # Parse the JSON response
        thread_data = response.json()

        # Update the last modified time for this thread
        if 'Last-Modified' in response.headers:
            last_modified_times[thread_id] = response.headers['Last-Modified']

        # Extract the original post (OP) and replies
        op_post = thread_data['posts'][0]  # First post is the OP
        op_text = op_post.get('com', 'No original post content.')

        replies = []
        for post in thread_data['posts'][1:]:  # Skip the first post (OP)
            reply_text = post.get('com', '')
            replies.append(reply_text)

        # Output the scraped data
        return {
            'thread_id': thread_id,
            'original_post': op_text,
            'replies': replies
        }

    except requests.RequestException as e:
        print(f"Error fetching thread {thread_id}: {e}")
        return None

# Function to get list of archived threads on the /lgbt/ board
def get_archived_threads():
    try:
        # Send request to get archived threads on the /lgbt/ board
        response = requests.get(archive_url)
        response.raise_for_status()

        # Parse the JSON response
        archived_thread_ids = response.json()

        # Return the thread IDs from the archive (limit to 100 threads)
        return archived_thread_ids[:100]  # Limit to 100 threads for scraping

    except requests.RequestException as e:
        print(f"Error fetching archived threads: {e}")
        return []

# Function to save scraped data to CSV
def save_to_csv(scraped_data, filename='scraped_archived_threads.csv'):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        # Write the header
        writer.writerow(['Thread ID', 'Post Type', 'Content'])

        # Write data for each thread
        for data in scraped_data:
            # Write the original post
            writer.writerow([data['thread_id'], 'Original Post', data['original_post']])

            # Write each reply
            for reply in data['replies']:
                writer.writerow([data['thread_id'], 'Reply', reply])

    print(f"Data saved to {filename}")

# Main function to scrape multiple archived threads and save to CSV
def scrape_archived_threads_and_save():
    # Get list of up to 100 archived thread IDs
    thread_ids = get_archived_threads()

    if not thread_ids:
        print("No threads found to scrape.")
        return

    # Scrape each thread and collect data
    all_scraped_data = []
    for thread_id in thread_ids:
        scraped_data = scrape_thread_api(thread_id)
        if scraped_data:
            all_scraped_data.append(scraped_data)

        # Delay to ensure no more than 1 request per second
        time.sleep(1)

    # Save the scraped data to a CSV file
    save_to_csv(all_scraped_data)

# Run the scraper for multiple archived threads and save the data
scrape_archived_threads_and_save()

# Add an update function for thread monitoring (optional)
def update_threads():
    while True:
        print("Updating archived threads...")
        scrape_archived_threads_and_save()

        # Wait for 10 seconds or more before updating again
        time.sleep(10)

# Uncomment below to start the update loop for continuous monitoring
# update_threads()


In [ ]:
import csv
from bs4 import BeautifulSoup

# Function to remove HTML tags from a string
def remove_html_tags(text):
    # Parse the text with BeautifulSoup and get only the text content
    return BeautifulSoup(text, 'html.parser').get_text()

# Function to process the CSV file
def clean_csv(input_csv, output_csv):
    with open(input_csv, mode='r', encoding='utf-8') as infile, \
         open(output_csv, mode='w', newline='', encoding='utf-8') as outfile:

        # Initialize CSV reader and writer
        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        # Iterate over rows in the input CSV
        for row in reader:
            # Clean each field in the row by removing HTML tags
            cleaned_row = [remove_html_tags(field) for field in row]
            # Write the cleaned row to the output CSV
            writer.writerow(cleaned_row)

    print(f"Data cleaned and saved to {output_csv}")

# Provide the input and output CSV file paths
input_csv = '/content/scraped_archived_threads.csv'  # Replace with your actual input CSV file
output_csv = 'scraped_archived_cleaned_data.csv'  # Replace with the desired output CSV file

# Call the function to clean the CSV
clean_csv(input_csv, output_csv)
